# Module 6 · Agentic RAG Testing
## The WidgetCo Story

**A scenario-driven walkthrough of every concept, golden, and metric in this module**

---

This notebook tells a story first, then builds everything from it.
By the end you will have:

| What | Why |
|---|---|
| A real imaginary product & company | So every test case has a *reason*, not just a label |
| The knowledge base the agent uses | So you know exactly what it can and cannot know |
| 6 story-backed test scenarios | One per golden dataset entry — each explained as a business situation |
| 4 custom metrics written from scratch | So you see why each check catches what off-the-shelf RAGAS metrics miss |
| A live coverage matrix | The same one from Module 4 Day 4, now with agentic failure columns |

> **Run order:** top to bottom. Live agent cells need `.env` configured in this `examples/` folder (see Day 1 setup note). Metric-building cells run fine without credentials.

---

## Slide 1 — Meet WidgetCo

**WidgetCo** is a B2B SaaS company that builds project-management and analytics tools for engineering teams. Founded 2019, originally based in Austin, TX.

### Timeline of events that matter for testing

| Year | What happened | Why it matters to us |
|---|---|---|
| 2019 | WidgetPro 2000 launches — WidgetCo's flagship product | Introduces a `$50` cancellation fee that will matter later |
| 2022 | HQ moves from Austin to Denver after Series B | Walk-in support policy changes — old info is still floating around |
| 2023 | WidgetPro 2000 discontinued → replaced by **WidgetPro 3000** | New product has `$0` cancellation fee — migrating customers keep asking about fees |
| 2024 | TurboMax 5 analytics add-on **rebranded to TurboMax Pro** | Same price (`$299/yr`), new name — confusion in support tickets spiked |
| Q1 2024 | **AI support agent deployed** to handle ticket surge from the 2000→3000 migration | The system we are testing |
| Q3 2024 | Support team flags recurring errors — agent citing wrong fees, wrong office policies | **Why WidgetCo hired us** |

### Why we're here

WidgetCo is planning a marketing campaign that will **3× inbound support volume**. Before that launch, they need their AI agent validated. We are the QA team. Our job: find every way the agent can give a wrong, misleading, or incomplete answer — and write tests that will catch those failures reliably.

The agent uses **agentic RAG**: it can retrieve from the knowledge base more than once before answering. That's what makes it powerful enough to handle complex policy questions — and what makes it possible for it to fail in ways a simple single-pass system never could.

---

## Slide 2 — The Knowledge Base

This is what the agent knows. Every correct answer must be traceable to one or more of these facts. Every fabrication is something the agent invented that isn't here.

Read it carefully — **the test scenarios we're about to write are designed around what's here AND what's missing**.

In [1]:
from agent import CORPUS

print('WidgetCo Knowledge Base — every fact the agent can retrieve')
print('=' * 60)
for i, fact in enumerate(CORPUS, 1):
    print(f'  [{i:02d}] {fact}')

print()
print('Notably ABSENT from this knowledge base:')
print('  - TurboMax Pro cancellation fee  (never documented — gap the agent must admit)')
print('  - WidgetPro 3000 annual price     (not in any policy doc yet)')

WidgetCo Knowledge Base — every fact the agent can retrieve
  [01] WidgetCo is a B2B SaaS company that builds project-management and analytics tools for engineering teams. Founded in 2019 and originally headquartered in Austin, TX.
  [02] Our headquarters relocated from Austin to Denver in 2022.
  [03] The Denver office does not offer walk-in support; all support is online only.
  [04] WidgetPro 2000 was discontinued in 2023 and replaced by WidgetPro 3000.
  [05] WidgetPro 3000's cancellation fee is $0 -- it can be canceled anytime at no charge.
  [06] WidgetPro 2000's cancellation fee was $50 before it was discontinued.
  [07] TurboMax 5 was renamed to TurboMax Pro in 2024 after a rebranding update.
  [08] TurboMax Pro's annual subscription costs $299, unchanged from TurboMax 5's price.
  [09] Our premium support plan includes 24/7 phone access and a 1-hour response SLA.

Notably ABSENT from this knowledge base:
  - TurboMax Pro cancellation fee  (never documented — gap the agent must

---

## Slide 3 — How the Agent Works

The agent isn't a simple lookup. It runs a **planner/retriever/generator loop**:

```
Customer question
      │
      ▼
  PLANNER ── 'Do I have enough to answer?' ──────────────► GENERATOR ► answer
      │  No                                         Yes
      │
      ▼
  'What should I search for next?'
      │
      ▼
  RETRIEVER (embedding similarity over the knowledge base)
      │
      └──────────────────────────── back to PLANNER
```

The loop adds power — AND new ways to fail:

| Failure mode | Plain English |
|---|---|
| **Premature stop** | Planner says 'enough' after 1 hop when the question needed 2 |
| **Reasoning chain break** | Both facts retrieved correctly, but combined wrong in the final answer |
| **Memory drop** | Fact from hop 1 does not survive to the final answer |
| **Ungraceful failure** | Agent fabricates an answer instead of admitting it does not know |

Each of the 6 scenarios we're about to build targets exactly one of these.

---

## Slide 4 — Our Testing Approach

Same three moves we've used since Module 4 Day 4, applied to each new failure mode:

1. **Tell the story** — what business situation produces this query?
2. **Write the golden** — what is the right answer, and what does a specific wrong answer look like? (golden + hard negative)
3. **Write the metric** — what function reliably distinguishes correct from incorrect for THIS failure mode?

We write our own metrics here — not because RAGAS is wrong, but because faithfulness, relevancy, recall, and precision were designed for a system that retrieves exactly once. Some of today's failure modes **cannot be detected by those metrics at all**. We'll prove that for the hardest case.

Let's run through all 6 scenarios.

---

## Scenario 1 — 'The Support Promise'
### `single-hop-support-01` · Failure mode: `hallucination`

**The business situation:**
A potential enterprise customer is evaluating WidgetCo's Premium Support tier. They ask the agent what the plan includes. The correct answer is in the knowledge base in one retrieval — 24/7 phone access and a 1-hour response SLA. Nothing else.

**What the failing agent does:**
Instead of stopping at what's documented, it adds a *dedicated account manager with same-day onsite visits* — which has never been offered and isn't in any policy doc.

**Why it matters:**
A sales prospect reads this, upgrades expecting onsite visits, then calls support to schedule one. Escalation. Refund request. Reputational damage. The business risk of a single hallucination in a support context isn't abstract.

**What makes this a single-hop problem:**
The knowledge base contains the full answer in one chunk. The planner doesn't need to retrieve more than once. The failure isn't in the loop — it's in the generator adding ungrounded claims.

In [2]:
import json

with open('golden_dataset.json') as f:
    ALL_GOLDENS = {c['id']: c for c in json.load(f)}

g  = ALL_GOLDENS['single-hop-support-01']
hn = ALL_GOLDENS['single-hop-support-01-hardneg']

print('GOLDEN — what a correct answer is graded against')
print(f'  question : {g["user_input"]}')
print(f'  reference: {g["reference"]}')
print()
print('HARD NEGATIVE — the specific wrong answer this test is designed to catch')
print(f'  retrieved context : {hn["retrieved_contexts"]}')
print(f'  fabricated answer : {hn["response"]}')
print()
print("Note: 'dedicated account manager' and 'same-day onsite visits' appear nowhere in the corpus.")

GOLDEN — what a correct answer is graded against
  question : What does our premium support plan include?
  reference: 24/7 phone access and a 1-hour response SLA.

HARD NEGATIVE — the specific wrong answer this test is designed to catch
  retrieved context : ['Our premium support plan includes 24/7 phone access and a 1-hour response SLA.']
  fabricated answer : Our premium support plan includes 24/7 phone access, a 1-hour response SLA, and a dedicated account manager with same-day onsite visits.

Note: 'dedicated account manager' and 'same-day onsite visits' appear nowhere in the corpus.


**Writing the metric:**

For hallucination on a single-hop case, the check is faithfulness — does every claim in the answer appear in the retrieved context? We write our own version here to see the logic plainly.

In [3]:
def keyword_faithfulness_check(answer: str, retrieved_contexts: list[str]) -> dict:
    """Every significant claim in the answer should be traceable to a retrieved chunk.
    Splits by sentence and checks each fragment against the joined context.
    This is the lightweight hand-rolled version of RAGAS faithfulness.
    """
    joined = ' '.join(retrieved_contexts).lower()
    claims = [s.strip() for s in answer.replace(',', '.').split('.') if len(s.strip()) > 10]
    grounded   = [c for c in claims if any(w.lower() in joined for w in c.split() if len(w) > 5)]
    ungrounded = [c for c in claims if c not in grounded]
    return {
        'passed': len(ungrounded) == 0,
        'grounded_claims': grounded,
        'ungrounded_claims': ungrounded,
        'score': len(grounded) / len(claims) if claims else 1.0,
    }


hn = ALL_GOLDENS['single-hop-support-01-hardneg']
result = keyword_faithfulness_check(hn['response'], hn['retrieved_contexts'])

print('Faithfulness check on the hard negative:')
print(f'  passed     : {result["passed"]}')
print(f'  score      : {result["score"]:.2f}')
print(f'  grounded   : {result["grounded_claims"]}')
print(f'  ungrounded : {result["ungrounded_claims"]}')

Faithfulness check on the hard negative:
  passed     : False
  score      : 0.67
  grounded   : ['Our premium support plan includes 24/7 phone access', 'a 1-hour response SLA']
  ungrounded : ['and a dedicated account manager with same-day onsite visits']


---

## Scenario 2 — 'Wrong Office, Wrong Era'
### `multi-hop-hq-01` · Failure mode: `hallucination` (multi-hop)

**The business situation:**
A WidgetCo customer in Denver is having an urgent issue and wants to know: does headquarters offer walk-in support visits? This sounds like a single-hop question — but it's not.

**Why two hops are needed:**
- Hop 1: What city is WidgetCo's current HQ in? (Austin → Denver, 2022)
- Hop 2: Does the Denver office accept walk-in visits? (No — online only)

**What the failing agent does:**
It retrieves the Denver relocation fact but then hallucinates that the Denver office welcomes walk-in visitors 9am–5pm on weekdays — something never true of Denver.

**Why it matters:**
A customer drives to the Denver office, finds no reception, calls support angry. Worse: a customer with a legal dispute now has a paper trail of the agent promising in-person access that does not exist.

In [ ]:
g  = ALL_GOLDENS['multi-hop-hq-01']
hn = ALL_GOLDENS['multi-hop-hq-01-hardneg']

print('GOLDEN')
print(f'  question : {g["user_input"]}')
print(f'  reference: {g["reference"]}')
print()
print('HARD NEGATIVE')
print(f'  hop 1 retrieved: {hn["retrieved_contexts"][0]}')
print(f'  hop 2 retrieved: {hn["retrieved_contexts"][1]}')
print(f'  fabricated answer: {hn["response"]}')
print()
print('Why faithfulness can miss this:')
print('  Denver appears in context → faithfulness treats Denver headquarters as grounded')
print('  But the walk-in claim contradicts the retrieved policy — faithfulness cannot detect contradiction')

In [ ]:
def location_policy_correct(answer: str) -> dict:
    """For the HQ walk-in question, a correct answer must:
    1. Mention Denver (the current HQ city, not Austin)
    2. Deny walk-in access (online-only policy)
    """
    answer_lower = answer.lower()
    mentions_denver = 'denver' in answer_lower
    denies_walkin = any(
        phrase in answer_lower
        for phrase in ['no walk', 'not offer walk', 'does not offer walk', 'online only', 'no in-person']
    )
    return {
        'passed': mentions_denver and denies_walkin,
        'mentions_current_city': mentions_denver,
        'correctly_denies_walkin': denies_walkin,
    }


correct_answer    = 'No -- headquarters relocated to Denver, and the Denver office does not offer walk-in support; all support is online only.'
fabricated_answer = 'Yes, our Denver headquarters welcomes walk-in visits every weekday from 9am to 5pm.'

for label, ans in [('CORRECT', correct_answer), ('HARD NEGATIVE', fabricated_answer)]:
    r = location_policy_correct(ans)
    print(f'[{label}]  passed={r["passed"]}  mentions_denver={r["mentions_current_city"]}  denies_walkin={r["correctly_denies_walkin"]}')

---

## Scenario 3 — 'The Name Chase'
### `multi-hop-turbomax-01` · Failure mode: `premature_stop`

**The business situation:**
A long-time WidgetCo customer knows the product they use as TurboMax 5 — the name it had when they bought it. They ask: what is the current annual subscription price for the product formerly known as TurboMax 5?

**Why two hops are needed:**
- Hop 1: Find out what TurboMax 5 is called now → TurboMax Pro
- Hop 2: Find TurboMax Pro's current annual price → $299

**What the failing agent does:**
After hop 1, the planner decides it knows enough. It generates: *TurboMax Pro is the current name for what used to be called TurboMax 5* — technically true, but completely nonresponsive to the actual question.

**The key insight:**
This answer is grounded (the rename fact is real) and relevant (it's about TurboMax 5). Standard faithfulness and answer relevancy checks can both PASS this hard negative. The only thing that catches it is checking whether the reference fact (the price) was ever retrieved at all.

In [ ]:
g  = ALL_GOLDENS['multi-hop-turbomax-01']
hn = ALL_GOLDENS['multi-hop-turbomax-01-hardneg']

print('GOLDEN')
print(f'  question : {g["user_input"]}')
print(f'  reference: {g["reference"]}')
print()
print('HARD NEGATIVE (premature_stop)')
print(f'  retrieved after 1 hop: {hn["retrieved_contexts"]}')
print(f'  planner stopped here. generated: {hn["response"]}')
print()
print('What this answer IS: grounded (rename fact is real), on-topic (about TurboMax 5)')
print('What this answer IS NOT: responsive — it never mentions a price')

In [ ]:
def price_retrieved(answer: str) -> dict:
    """For pricing questions, the answer must contain a dollar amount.
    An on-topic answer without a price is a premature_stop in disguise.
    """
    import re
    prices_found = re.findall(r'\$\d+', answer)
    return {
        'passed': len(prices_found) > 0,
        'prices_found': prices_found,
        'note': 'No price in answer — planner likely stopped before retrieving the price fact' if not prices_found else 'Price present',
    }


correct_answer  = 'TurboMax Pro annual subscription costs $299.'
premature_answer = 'TurboMax Pro is the current name for what used to be called TurboMax 5.'

for label, ans in [('CORRECT', correct_answer), ('PREMATURE STOP', premature_answer)]:
    r = price_retrieved(ans)
    print(f'[{label}]  passed={r["passed"]}  prices={r["prices_found"]}')
    print(f'   {r["note"]}')

---

## Scenario 4 — 'The Memory Drop'
### `multi-hop-widgetpro-01-memory-drop` · Failure mode: `reasoning_chain_break` (memory variant)

**The business situation:**
A WidgetCo customer migrated from WidgetPro 2000 wants to cancel. They ask for the cancellation fee on the replacement product. Two hops required: identify the replacement product (WidgetPro 3000), then retrieve its fee ($0).

**What the failing agent does:**
It finds the right fee ($0) but drops the product name from hop 1. It answers: *The cancellation fee is $0.*

**Why this matters:**
The fee is correct but the answer is now ambiguous. A WidgetPro 2000 customer who doesn't know which product the $0 applies to may misread it as applying to their own (discontinued) product.

**The testing insight:**
This is Module 5 Day 2's chunk-boundary bug relocated to conversation memory. The boundary is wherever the agent truncates its running context — and it can drop a needed fact the same way a 500-token document boundary could.

In [ ]:
mc = ALL_GOLDENS['multi-hop-widgetpro-01-memory-drop']

print('GOLDEN (memory-drop hard negative)')
print(f'  question     : {mc["user_input"]}')
print(f'  reference    : {mc["reference"]}')
print(f'  must_include : {mc["must_include"]}')
print()
print(f'  actual answer: {mc["response"]}')
print()
print('Problem: \$0 is correct. But WidgetPro 3000 — established in hop 1 — is gone from the answer.')

In [ ]:
def memory_intact(answer: str, must_include: list[str]) -> dict:
    """All key facts from early hops must survive into the final answer.
    must_include is the list of terms the golden requires to be present.
    A missing term means a hop's finding was dropped during generation.
    """
    missing = [term for term in must_include if term not in answer]
    return {
        'passed': len(missing) == 0,
        'missing_facts': missing,
        'note': f'Hop-1 fact(s) dropped: {missing}' if missing else 'All required facts present',
    }


correct_answer    = 'WidgetPro 3000 cancellation fee is $0 — it replaced WidgetPro 2000 in 2023.'
memory_drop_answer = 'The cancellation fee is $0.'
required = ['WidgetPro 3000']

for label, ans in [('CORRECT', correct_answer), ('MEMORY DROP', memory_drop_answer)]:
    r = memory_intact(ans, required)
    print(f'[{label}]  passed={r["passed"]}')
    print(f'   {r["note"]}')

---

## Scenario 5 — 'The Bait-and-Switch Fee'
### `multi-hop-widgetpro-01-hardneg` · Failure mode: `reasoning_chain_break` (combination error)

**The business situation:**
Same question: what is the cancellation fee for the product that replaced WidgetPro 2000?

**What makes this scenario different from Scenario 4:**
The agent retrieves BOTH relevant facts correctly:
- Fact A: WidgetPro 2000 was discontinued and replaced by WidgetPro 3000
- Fact B: WidgetPro 2000's cancellation fee was $50

Both facts are individually true. Both are grounded in the knowledge base. Then the agent writes: *The cancellation fee for the product that replaced WidgetPro 2000 is $50* — attaching the OLD product's fee to the REPLACEMENT product.

**This is the hardest failure mode in this module:**

| Check | Result | Reason |
|---|---|---|
| `faithfulness` | **PASSES** | $50 is in the retrieved context — it's grounded |
| `answer_relevancy` | **PASSES** | The answer is on-topic about the right question |
| `context_precision` | **PASSES** | The retrieved chunks are relevant |
| `memory_intact` | **PASSES** | WidgetPro 3000 IS in the answer |
| `reasoning_chain_correct` | **FAILS** | The wrong product's fee was applied |

This failure mode is **structurally invisible to per-fact checks**. It only becomes visible when you check which fact was actually *applied* to which product.

In [ ]:
g  = ALL_GOLDENS['multi-hop-widgetpro-01']
hn = ALL_GOLDENS['multi-hop-widgetpro-01-hardneg']

print('GOLDEN (correct answer)')
print(f'  question        : {g["user_input"]}')
print(f'  reference       : {g["reference"]}')
print(f'  must_include    : {g["must_include"]}')
print(f'  must_not_include: {g["must_not_include"]}')
print()
print('HARD NEGATIVE (reasoning_chain_break)')
print(f'  retrieved fact 1: {hn["retrieved_contexts"][0]}')
print(f'  retrieved fact 2: {hn["retrieved_contexts"][1]}')
print(f'  wrong answer    : {hn["response"]}')
print()
print('Both facts are in the knowledge base. Both were retrieved. The COMBINATION is the bug.')

In [ ]:
def reasoning_chain_correct(
    answer: str,
    must_include: list[str],
    must_not_include: list[str],
) -> dict:
    """Full reasoning-combination check.
    must_include: facts the correct answer must contain
    must_not_include: facts that signal the wrong combination
    """
    missing       = [t for t in must_include     if t not in answer]
    wrongly_present = [t for t in must_not_include if t in answer]
    passed = len(missing) == 0 and len(wrongly_present) == 0
    return {
        'passed': passed,
        'missing_required': missing,
        'wrong_terms_present': wrongly_present,
        'note': 'Correct combination' if passed else f'Reasoning error — missing={missing}, wrong_present={wrongly_present}',
    }


hn = ALL_GOLDENS['multi-hop-widgetpro-01-hardneg']
correct_answer = 'WidgetPro 3000 cancellation fee is $0.'
wrong_answer   = hn['response']

for label, ans in [('CORRECT', correct_answer), ('REASONING BREAK', wrong_answer)]:
    r = reasoning_chain_correct(ans, hn['must_include'], hn['must_not_include'])
    print(f'[{label}]  passed={r["passed"]}')
    print(f'   {r["note"]}')

print()
print('Now prove that faithfulness PASSES the hard negative:')
faith = keyword_faithfulness_check(wrong_answer, hn['retrieved_contexts'])
print(f'  faithfulness passed={faith["passed"]}  score={faith["score"]:.2f}')
print('  → faithfulness sees $50 is in context and calls it grounded. Correct — and useless here.')

---

## Scenario 6 — 'The Phantom Policy'
### `graceful-failure-01` · Failure mode: `ungraceful_failure`

**The business situation:**
A TurboMax Pro customer wants to understand their cancellation options before the renewal date. They ask: what is the cancellation fee for TurboMax Pro?

**The hard part:**
WidgetCo never documented TurboMax Pro's cancellation fee. It simply isn't in the knowledge base — not as $0, not as $50, not as anything. There is no correct number to give.

**What the failing agent does:**
After several hops find nothing specific, the generator fills the gap: *The cancellation fee for TurboMax Pro is $25* — a number that was fabricated outright.

**What the correct agent should do:**
Admit the gap. *I wasn't able to find TurboMax Pro's cancellation fee in our documentation. Please contact support directly.* That's not a failure of intelligence — it's a sign of trustworthiness.

**Why this matters most:**
This is the failure pattern Klarna's incident report pointed at: agents that fabricate to keep a conversation moving rather than admitting a knowledge gap. **Graceful failure is testable. It's not a vibe.**

In [ ]:
g  = ALL_GOLDENS['graceful-failure-01']
hn = ALL_GOLDENS['graceful-failure-01-hardneg']

print('GOLDEN')
print(f'  question  : {g["user_input"]}')
print(f'  reference : {g["reference"]}')
print()
print('HARD NEGATIVE (ungraceful_failure)')
print(f'  fabricated answer: {hn["response"]}')
print()
print('Corpus check — does $25 appear anywhere in the knowledge base?')
from agent import CORPUS
matches = [f for f in CORPUS if '25' in f]
print(f'  Corpus entries containing 25: {matches or "(none)"}')
print('  → $25 is completely fabricated. No source for this number exists anywhere.')

In [ ]:
def graceful_failure(answer: str) -> dict:
    """When the knowledge base has no answer, the agent should hedge rather than fabricate.
    A passing answer contains at least one hedging phrase signaling uncertainty.
    """
    hedge_phrases = [
        "don't have",
        "doesn't have",
        "couldn't find",
        'no information',
        'not available',
        'not in the knowledge base',
        'unable to find',
        'not documented',
        'please contact',
        'reach out to support',
    ]
    answer_lower  = answer.lower()
    hedges_found  = [p for p in hedge_phrases if p in answer_lower]
    return {
        'passed': len(hedges_found) > 0,
        'hedges_found': hedges_found,
        'note': f'Graceful — admitted gap with: {hedges_found}' if hedges_found else 'Ungraceful — answered without hedging (possible fabrication)',
    }


graceful_answer   = "I wasn't able to find TurboMax Pro's cancellation fee in our documentation. Please contact support directly."
ungraceful_answer = 'The cancellation fee for TurboMax Pro is $25.'

for label, ans in [('GRACEFUL', graceful_answer), ('UNGRACEFUL (hard negative)', ungraceful_answer)]:
    r = graceful_failure(ans)
    print(f'[{label}]  passed={r["passed"]}')
    print(f'   {r["note"]}')

---

## Slide 7 — Running the Real Agent Against All 6 Scenarios

Now we run the actual agentic RAG loop (real LLM planner, real embedding retrieval, real generator) against each of the golden questions and score them with our custom metrics.

**What to watch for:**
- `num_hops` — how many retrieval cycles did it take?
- `hit_max_hops` — did the planner confidently stop, or exhaust the hop limit?
- Which custom metric catches which failure, and which cases are edge-call?

> **Reminder:** You need `.env` configured in this `examples/` folder with your LLM credentials. If you don't have credentials, the metric cells above are fully self-contained and show the logic on scripted examples.

In [ ]:
from agent import agentic_rag

# Map each scenario to its metric function.
SCENARIOS = [
    ('single-hop-support-01',
     'What does our premium support plan include?',
     lambda ans, r: keyword_faithfulness_check(ans, ['Our premium support plan includes 24/7 phone access and a 1-hour response SLA.']),
     'keyword_faithfulness_check'),
    ('multi-hop-hq-01',
     'Does our headquarters offer walk-in support visits?',
     lambda ans, r: location_policy_correct(ans),
     'location_policy_correct'),
    ('multi-hop-turbomax-01',
     'What is the current annual subscription price for the product formerly known as TurboMax 5?',
     lambda ans, r: price_retrieved(ans),
     'price_retrieved'),
    ('multi-hop-widgetpro-01',
     'What is the cancellation fee for the product that replaced WidgetPro 2000?',
     lambda ans, r: reasoning_chain_correct(ans, ['WidgetPro 3000', '$0'], ['$50']),
     'reasoning_chain_correct'),
    ('graceful-failure-01',
     'What is the cancellation fee for TurboMax Pro?',
     lambda ans, r: graceful_failure(ans),
     'graceful_failure'),
]

print(f'{len(SCENARIOS)} scenarios queued.')

In [ ]:
results_table = []

for golden_id, question, metric_fn, metric_label in SCENARIOS:
    print(f'\n{"="*60}')
    print(f'Scenario: {golden_id}')
    print(f'Question: {question}')

    result = await agentic_rag(question, verbose=False)
    metric_result = metric_fn(result.response, result)

    print(f'  hops={result.num_hops}  hit_max={result.hit_max_hops}')
    print(f'  answer: {result.response}')
    print(f'  [{metric_label}] passed={metric_result["passed"]}')

    results_table.append({
        'id': golden_id, 'hops': result.num_hops,
        'hit_max_hops': result.hit_max_hops,
        'metric': metric_label, 'passed': metric_result['passed'],
    })

print(f'\n{"="*60}')
print('\nSUMMARY TABLE')
print(f'{"Scenario":<40} {"Hops":>4}  {"Max?":>5}  {"Passed":>6}')
for r in results_table:
    print(f'{r["id"]:<40} {r["hops"]:>4}  {str(r["hit_max_hops"]):>5}  {str(r["passed"]):>6}')

---

## Slide 8 — The Coverage Matrix

The coverage matrix from Module 4 Day 4, extended twice: Module 5 Day 2 added retrieval columns; today we add agentic failure columns. The matrix now shows all 6 WidgetCo scenarios.

In [ ]:
import json
from collections import Counter

with open('golden_dataset.json') as f:
    golden_cases = json.load(f)

categories    = sorted({c['category']     for c in golden_cases})
failure_modes = sorted({c['failure_mode'] for c in golden_cases})
counts        = Counter((c['category'], c['failure_mode']) for c in golden_cases)

col_w = 24
header = f'{"":20}' + ''.join(f'{fm:<{col_w}}' for fm in failure_modes)
print(header)
print('-' * len(header))
for cat in categories:
    row = f'{cat:<20}' + ''.join(
        f'{counts[(cat, fm)] if counts[(cat, fm)] else chr(8212):<{col_w}}'
        for fm in failure_modes
    )
    print(row)

print()
print('Reading the matrix:')
print('  single_hop_qa — agentic columns are empty not because we forgot them,')
print('                  but because premature_stop / reasoning_chain_break / ungraceful_failure')
print('                  cannot occur when a question requires only 1 hop.')
print('  multi_hop_qa  — every agentic failure mode has at least 2 cases (golden + hard negative).')
print()
print('Uncovered gap (from Day 1 failure-mode table):')
print('  query_drift — no column yet. That gap is the Try It Yourself exercise for Day 2.')

---

## Slide 9 — What We Built and Why

### The 4 custom metrics

| Metric | Catches | Why not just RAGAS? |
|---|---|---|
| `keyword_faithfulness_check` | Generator added claims not in retrieved context | RAGAS faithfulness does the same — this shows the logic; use RAGAS in production |
| `location_policy_correct` | Multi-hop hallucination: real location, fabricated policy | RAGAS faithfulness misses contradictions when entity names appear in context |
| `price_retrieved` | Premature stop — agent stopped before retrieving the actual answer | RAGAS context_recall would also catch this; `price_retrieved` is simpler and transparent |
| `memory_intact` | Hop-1 facts dropped from the final answer | No RAGAS metric checks what made it into the *final answer* vs what was *retrieved* |
| `reasoning_chain_correct` | Right facts, wrong combination | **Faithfulness structurally cannot catch this** — the wrong claim IS grounded in context |
| `graceful_failure` | Fabrication when knowledge base has no answer | No RAGAS metric scores graceful admission of a knowledge gap |

### The pattern

Every metric followed the same 3 steps: tell the story → write the golden + hard negative → write the metric. The same loop that's been running since Module 4 Day 4, now applied to failure modes that only exist once retrieval can happen more than once.

---

**Next:** Day 1 notebook (`01_multistep_retrieval.ipynb`) runs the full planner/executor loop with LangSmith tracing and digs into how the planner's reasoning looks hop-by-hop — the mechanism behind the scenarios we just described from the outside.